In [2]:
from routing_investigate import *
from routing import *
import arviz as az

In [ ]:
data = az.from_netcdf("path/to/your_file.nc")

In [ ]:
pi_draws, A_draws, mu_draws, sigma_draws = (extract_posterior_draws(aligned_trace))

In [ ]:
print(pi_draws.shape, A_draws.shape, mu_draws.shape, sigma_draws.shape)

In [ ]:
sequence = val_sequences[0]

results = filter_posterior_draws(
    sequence,
    pi_draws,
    A_draws,
    mu_draws,
    sigma_draws,
)

In [ ]:
print(results["filtered_probs"].shape)
print(results["uncertainty"].shape)
print(results["surprise"].shape)
print(results["information_gain"].shape)

In [ ]:
print(
    results["filtered_probs"]
    .sum(axis=1)[:10]
)

In [ ]:
print(
    results["uncertainty"].min(),
    results["uncertainty"].max(),
)

In [ ]:
print(
    results["parameter_disagreement"].min()
)

In [ ]:
account_id = val_df["account_id"].unique()[0]

In [ ]:
probs = results["filtered_probs"]

In [ ]:
routing_df = pd.DataFrame({
        "account_id": account_id,
        "time": np.arange(len(probs)),

        "p_state_0": probs[:, 0],
        "p_state_1": probs[:, 1],
        "p_state_2": probs[:, 2],

        "uncertainty": results["uncertainty"],
        "surprise": results["surprise"],
        "information_gain": results["information_gain"],

        "expected_state_entropy":
            results["expected_state_entropy"],

        "parameter_disagreement":
            results["parameter_disagreement"],
    })

In [ ]:
routing_df.head()

In [ ]:
plt.rcParams["figure.figsize"] = (10, 5)
plt.plot(routing_df["surprise"], label="suprise")
plt.plot(routing_df["uncertainty"], label= "uncertainty")
plt.legend()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(routing_df["time"], routing_df["surprise"])
axes[0].set_ylabel("Predictive surprise")

axes[1].plot(routing_df["time"], routing_df["uncertainty"])
axes[1].set_ylabel("Normalized entropy")
axes[1].set_xlabel("Time")

plt.tight_layout()
plt.show()

In [ ]:
account_id = routing_df["account_id"].iloc[0]

truth = (
    val_df[val_df["account_id"] == account_id]
    [["time", "aml_state"]]
    .sort_values("time")
)

comparison = routing_df.merge(
    truth,
    on="time",
    validate="one_to_one",
)

comparison["state_transition"] = (
    comparison["aml_state"]
    .ne(comparison["aml_state"].shift())
)

comparison.loc[comparison.index[0], "state_transition"] = False

In [ ]:
transition_times = comparison.loc[
    comparison["state_transition"],
    "time"
].to_numpy()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, metric in zip(axes, ["surprise", "uncertainty"]):
    ax.plot(comparison["time"], comparison[metric])

    for t in transition_times:
        ax.axvline(t, color="red", alpha=0.25, linestyle="--")

    ax.set_ylabel(metric)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
validation_routing_df = build_routing_dataframe(
    val_df,
    val_sequences,
    pi_draws,
    A_draws,
    mu_draws,
    sigma_draws,
)

print(validation_routing_df.shape)
print(validation_routing_df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))

plt.scatter(
    validation_routing_df["uncertainty"],
    validation_routing_df["surprise"],
    alpha=0.3,
    s=15,
)

plt.xlabel("Normalized uncertainty")
plt.ylabel("Predictive surprise")
plt.title("Validation accounts: uncertainty vs surprise")

plt.show()

In [ ]:
transition_times

In [ ]:
from scipy.stats import spearmanr, pearsonr

u = validation_routing_df["uncertainty"]
s = validation_routing_df["surprise"]eran

print("Pearson:", pearsonr(u, s))
print("Spearman:", spearmanr(u, s))

In [ ]:
from routing_investigate import (
    build_candidate_windows,
    select_investigations,
    validate_investigations,
)

candidates = build_candidate_windows(
    validation_routing_df,
    window_size=10,
)

print(candidates.shape)
print(candidates.head())

In [ ]:
strategies = [
    "uncertainty",
    "surprise",
    "combined",
    "random",
]

selected_windows = {}

for strategy in strategies:

    selected = select_investigations(
        candidates,
        budget=10,
        strategy=strategy,
        random_state=42,
    )

    validate_investigations(
        selected,
        candidates,
        budget=10,
    )

    selected_windows[strategy] = selected

    print(f"\nStrategy: {strategy}")
    print(
        selected[
            [
                "account_id",
                "start_time",
                "end_time",
                "routing_score",
            ]
        ].to_string(index=False)
    )

In [ ]:
def selection_overlap(selected_a, selected_b):
    """
    Number of identical selected windows.
    """

    keys_a = set(
        zip(selected_a["account_id"], selected_a["end_time"])
    )

    keys_b = set(
        zip(selected_b["account_id"], selected_b["end_time"])
    )

    return len(keys_a & keys_b)


for i, strategy_a in enumerate(strategies):

    for strategy_b in strategies[i + 1:]:

        overlap = selection_overlap(
            selected_windows[strategy_a],
            selected_windows[strategy_b],
        )

        print(
            f"{strategy_a} vs {strategy_b}: "
            f"{overlap} identical windows"
        )

In [ ]:
print("Validation accounts:", val_df["account_id"].nunique())
print("Routing accounts:", validation_routing_df["account_id"].nunique())
print("Candidate accounts:", candidates["account_id"].nunique())

print("Validation observations:", len(val_df))
print("Candidate windows:", len(candidates))

assert set(candidates["account_id"]) == set(val_df["account_id"])

In [ ]:
def temporal_overlap(selected_a, selected_b):
    """
    Count how many selected windows in A overlap
    at least one selected window in B.
    """

    count = 0

    for _, a in selected_a.iterrows():

        overlaps = selected_b[
            (selected_b["account_id"] == a["account_id"])
            & (selected_b["start_idx"] <= a["end_idx"])
            & (selected_b["end_idx"] >= a["start_idx"])
        ]

        if not overlaps.empty:
            count += 1

    return count

In [ ]:
for i, strategy_a in enumerate(strategies):
    for strategy_b in strategies[i + 1:]:

        overlap = temporal_overlap(
            selected_windows[strategy_a],
            selected_windows[strategy_b],
        )

        print(
            f"{strategy_a} vs {strategy_b}: "
            f"{overlap}/10 windows overlap temporally"
        )